In [ ]:
"""
Space Themed RPG with Inventory Detection
Simplest possible setup so far, more will be added in the future.

Features:
- Can load any planet file
- Plays with one character (selected from a character list)
- Inventory detection
- The character has a set amount of health, if it hits zero, the character dies and must restart.
"""

import gradio as gr
from langchain_community.llms import Ollama
import json
import glob

# Initialize LLM
llm = Ollama(model="mistral", temperature=0.74)

In [ ]:
# ============================================================================
# GAME STATE
# ============================================================================

GAME_STATE = {
    'planet': None,
    'region': None,
    'town': None,
    'character': None,
    'inventory': {},  # {item_name: quantity}
    'started': False,
    'health': 100, # health starts full
    'max_health': 100,
    'dead': False # controls if character is dead or not
}

def load_planet_file(filename: str):
    """Load a pre-generated world from JSON."""
    with open(filename, 'r') as f:
        data = json.load(f)
    
    # Handle metadata wrapper if present
    if 'planet_data' in data:
        return data['planet_data']
    return data


In [ ]:
# ============================================================================
# SETUP FUNCTION
# ============================================================================

def setup_game(planet_filename: str, character_name: str) -> str:
    """
    Simple setup - load world and pick a character.
    """
    try:
        GAME_STATE['inventory'] = [] # resets inventory
        GAME_STATE['health'] = 100 # resets health
        
        # Load the world
        planet_data = load_planet_file(planet_filename)
        
        GAME_STATE['planet'] = planet_data['planet']
        GAME_STATE['region'] = planet_data['regions'][0]
        GAME_STATE['town'] = GAME_STATE['region']['towns'][0]

        # Find the character by name
        found_character = None
        for region in planet_data['regions']:
            for town in region['towns']:
                for npc in town.get('npcs', []):
                    if character_name.lower() in npc['name'].lower():
                        found_character = npc
                        GAME_STATE['character'] = npc
                        GAME_STATE['region'] = region
                        GAME_STATE['town'] = town
                        GAME_STATE['inventory'] = { #starting inventory
                            "laser gun": 1,
                            "health potion": 2,
                            "credits": 100
                        }
                        break
                if found_character:
                    break
            if found_character:
                break
        
        if not found_character:
            return f" Character '{character_name}' not found!"
        
        return f"""  Game loaded!

Planet: {GAME_STATE['planet']['name']}
Playing as: {GAME_STATE['character']['name']} ({GAME_STATE['character']['role']}), your race is a {GAME_STATE['character']['race']}
Location: {GAME_STATE['town']['name']}, {GAME_STATE['region']['name']}

Type 'start game' to begin!"""
        
    except FileNotFoundError:
        return f" World file '{world_filename}' not found!\nGenerate one first: python rpg_content_generator.py"
    except Exception as e:
        return f" Error: {str(e)}"


In [ ]:
# ============================================================================
# INVENTORY DETECTION
# ============================================================================

def detect_inventory_changes(story: str, current_inventory: dict) -> list:
    """Use LLM to detect what items changed in the story."""
    
    system_prompt = """You are an AI Game Assistant. \
Your job is to detect changes to a player's \
inventory based on the most recent story and game state.

If a player picks up, or is given an item add it to the inventory \
with a positive change_amount.
If a player loses an item remove it from their inventory \
with a negative change_amount.

Given a player inventory and story, return a list of json updates \
of the player's inventory in the following form.

Only take items that it's clear the player (you) lost.
laser gun cannot be lost.
health potions cannot be lost unless the player drinks them.
Only give items that it's clear the player gained. 
health is not an item. 
Only remove health potions if they are drank.
Do not use a health potion if health is at zero or lower.
Do not take items that are with another character.
Don't make any other item updates.
If no items were changed return {"itemUpdates": []}
and nothing else.

Response must be in Valid JSON.
Don't add items that were already added in the inventory.

Inventory Updates:
{
    "itemUpdates": [
        {"name": <ITEM NAME>, 
        "change_amount": <CHANGE AMOUNT>}...
    ]
}
"""
    
    # Format current inventory
    inventory_str = ", ".join([f"{name} (x{qty})" for name, qty in current_inventory.items()])
    if not inventory_str:
        inventory_str = "Empty"
    
    prompt = f"""{system_prompt}

Current Inventory: {inventory_str}

Recent Story:
{story}

JSON Response:"""
    
    try:
        response = llm.invoke(prompt)        
        # Extract JSON from response
        json_start = response.find('{')
        json_end = response.rfind('}') + 1
        
        if json_start != -1 and json_end > json_start:
            json_str = response[json_start:json_end]
            print(f"Extracted JSON: {json_str}")
            updates = json.loads(json_str)
            print(f"Parsed updates: {updates.get('itemUpdates', [])}")
            return updates.get('itemUpdates', [])
        
        print("No valid JSON found in response")
        return []
        
    except Exception as e:
        print(f"Inventory detection error: {e}")
        print(f"Response was: {response if 'response' in locals() else 'No response'}")
        return []


def update_inventory(item_name: str, change_amount: int):
    """Update the inventory dictionary."""    
    if item_name in GAME_STATE['inventory']:
        GAME_STATE['inventory'][item_name] += change_amount
        if GAME_STATE['inventory'][item_name] <= 0:
            del GAME_STATE['inventory'][item_name]
    elif change_amount > 0:
        GAME_STATE['inventory'][item_name] = change_amount


def get_inventory_string() -> str:
    """Format inventory for display."""
    if not GAME_STATE['inventory']:
        return " Your inventory is empty."
    
    items = [f"{name} (x{qty})" for name, qty in GAME_STATE['inventory'].items()]
    return " Inventory: " + ", ".join(items)

In [ ]:
#=============================================================================
# HEALTH SYSTEM
#=============================================================================

def detect_health_changes(story: str, health: int) -> list:
    """Use LLM to detect health changes within the story."""

    system_prompt = """You are an AI Game Assistant. \
Your job is to detect changes to a player's \
health based on the recent story and game state.

If a player fights an enemy, lower the health value by a random value between 10 and 99. Health should not be lowered when first encountering the enemy.
If a player drinks a health potion, reset their health to the value of max health using positive change_amount.
Do not drink a health potion if health is at zero or below.

Given a player's health and story, return a list of json updates of the player's health in the following form:

Only lower health when it's clear it should be lost.
Only add health when a health potion is clearly drank.
Health can not be larger than the value of 100.

If no health was changed return {"healthUpdates": []} and nothing else.

Response must be in a valid JSON.

Health Updates:
{
    "healthUpdates": [
        "change_amount":<CHANGE AMOUNT>}.....
    ]
}
"""

    prompt = f"""{system_prompt}

Current Health: {health}

Recent Story: 
{story}

JSON Response:"""
    
    try:
        response = llm.invoke(prompt)        
        # Extract JSON from response
        json_start = response.find('{')
        json_end = response.rfind('}') + 1
        
        if json_start != -1 and json_end > json_start:
            json_str = response[json_start:json_end]
            print(f"Extracted JSON: {json_str}")
            updates = json.loads(json_str)
            print(f"Parsed updates: {updates.get('healthUpdates', [])}")
            return updates.get('healthUpdates', [])
        
        print("No valid JSON found in response")
        return []
        
    except Exception as e:
        print(f"Inventory detection error: {e}")
        print(f"Response was: {response if 'response' in locals() else 'No response'}")
        return []

def update_health(change_amount: int):
    """Update the value of health."""    
    if change_amount < 0:
        GAME_STATE['health'] += change_amount
    if GAME_STATE['health'] < 0:
        GAME_STATE['dead'] = True
    if change_amount > 0:
        GAME_STATE['health'] = GAME_STATE['max_health']

def get_health_string() -> str:
    """Format health for display."""
    return f" Health: {GAME_STATE['health']}"

In [ ]:
# ============================================================================
# GAME LOOP
# ============================================================================

def run_action(message: str, history: list) -> str:
    """Main game loop"""

    if GAME_STATE['dead'] == True: # Nothing else can be done if dead
        return "You are dead, you must restart the adventure."
    
    # Command: inventory
    if message.lower().strip() == 'inventory':
        return get_inventory_string()
    
    # Command: start game 
    if message.lower().strip() == 'start game':
        if not GAME_STATE['character']:
            return " No planet loaded! Run setup first."
        
        GAME_STATE['started'] = True
        
        # Create opening scene with LLM
        system_prompt = """You are an AI Game master. Your job is to create a 
start to an adventure based on the planet, region, town and character 
a player is playing as. 

Instructions:
- You must only use 2-4 sentences
- Write in second person. For example: "You are Kyrion"
- Write in present tense. For example "You stand at..."
- First describe the character and their backstory
- Then describe where they start and what they see around them"""

        planet_info = f"""
Planet: {GAME_STATE['planet']['name']} - {GAME_STATE['planet']['description']}
Region: {GAME_STATE['region']['name']}
Town: {GAME_STATE['town']['name']} - {GAME_STATE['town']['description']}
Your Character: {GAME_STATE['character']['name']}, {GAME_STATE['character']['role']} {GAME_STATE['character']['race']} - {GAME_STATE['character']['description']}

Starting Health: {get_health_string()}
Starting Inventory: {get_inventory_string()}
"""
        
        prompt = f"{system_prompt}\n\n{planet_info}\n\nGame Master:"
        
        try:
            response = llm.invoke(prompt)
            return response.strip() + f"\n{get_health_string()}\n{get_inventory_string()}" # Shows health and inventory
        except Exception as e:
            return f" Error: {str(e)}\nIs Ollama running?"
    
    # Check if game started
    if not GAME_STATE['started']:
        return "Type 'start game' to begin!"
    
    # Main game loop
    system_prompt = """You are an AI Game Master. Your job is to write what 
happens next in a player's adventure game.

Instructions:
- Write ONLY 2-3 sentences in response
- Always write in second person present tense, say you instead of the name of the player character
- Be creative and make the story interesting
- focus on danger and conflict
- be aware of death when it occurs.
- Do not let the player speak to the character they have chosen when talking to nearby npcs
- Do not use a health potion if your health is at zero or below.
- do not use any questions to end your responses.
- Do not predict the player."""

    planet_info = f"""
Planet: {GAME_STATE['planet']['name']}
Town: {GAME_STATE['town']['name']}
Character: {GAME_STATE['character']['name']}, a {GAME_STATE['character']['role']}
Current Health: {get_health_string()}
Current Inventory: {get_inventory_string()}
"""
    
    full_prompt = f"{system_prompt}\n\n{planet_info}\n\n"
    
    # Add history
    for user_msg, bot_msg in history[-5:]:  # Last 5 turns
        full_prompt += f"Player: {user_msg}\n"
        full_prompt += f"Game Master: {bot_msg}\n\n"
    
    full_prompt += f"Player: {message}\n\nGame Master:"
    
    # Get GM response
    try:
        response = llm.invoke(full_prompt)
        response = response.strip()

        # HEALTH DETECTION
        print(f"\nTURN START - Current health: {GAME_STATE['health']}")
        story_context = f"Player action: {message}\nGame Master response: {response}"
        health_updates = detect_health_changes(story_context, GAME_STATE['health'])

        # Apply updates
        health_messages = []
        if health_updates:
            print(f" Applying {len(health_updates)} health updates")
            for update in health_updates:
                change_amount = update.get('change_amount', 0)

                if change_amount != 0:
                    print(f" -> Updating health by {change_amount}")
                    update_health(change_amount)

                    if change_amount > 0:
                        health_messages.append(f"❤️ Health has been restored to 100!!")
                    else:
                        health_messages.append(f"💔 Lost {abs(change_amount)} health")
        else:
            print("No health changes detected.")
        
        # INVENTORY DETECTION
        story_context = f"Player action: {message}\nGame Master response: {response}"
        item_updates = detect_inventory_changes(story_context, GAME_STATE['inventory'])
        
        # Apply updates
        inventory_messages = []
        if item_updates:
            print(f" Applying {len(item_updates)} inventory updates")
            for update in item_updates:
                item_name = update.get('name', '')
                change_amount = update.get('change_amount', 0)
                
                if item_name and change_amount != 0:
                    print(f"  → Updating: {item_name} by {change_amount}")
                    update_inventory(item_name, change_amount)
                    
                    if change_amount > 0:
                        inventory_messages.append(f"✨ Gained: {item_name} (x{change_amount})")
                    else:
                        inventory_messages.append(f"❌ Lost: {item_name} (x{abs(change_amount)})")
        else:
            print(" No inventory changes detected")
        
        print(f" TURN END - Final inventory: {GAME_STATE['inventory']}, Final health: {GAME_STATE['health']}")
        
        # Build response
        final_response = response

        if health_messages:
            final_response += "\n\n" + "\n".join(health_messages)
            final_response += f"\n{get_health_string()}"
        
        if inventory_messages:
            final_response += "\n\n" + "\n".join(inventory_messages)
            final_response += f"\n{get_inventory_string()}"

        if GAME_STATE['dead']:
            final_response += "\n\nYou have died."
        
        return final_response
        
    except Exception as e:
        return f" Error: {str(e)}"

In [ ]:
# ============================================================================
# GRADIO INTERFACE
# ============================================================================

with gr.Blocks(theme=gr.themes.Soft(), fill_height=True) as interface:
    
    gr.Markdown("""
    # Space Themed RPG
    
    **Setup Steps:**
    1. Enter your world filename (must exist!)
    2. Enter a character name from that world
    3. Click "Load Game"
    4. Type `start game` in chat
    """)
    
    with gr.Row():
        planet_input = gr.Textbox(
            label="Planet File",
            placeholder="zephyrion9.json",
            value="zephyrion9.json"  # Example default
        )
        character_input = gr.Dropdown(
            choices=["Galaxor Zyrion", "Nebula Nightshade", "Gritto the Sandy Tinkerer", "Kyrion Frostbane"],
            label="Character Name",
            multiselect=False,
            value="Kyrion Frostbane"
        )
    
    load_btn = gr.Button(" Load Game", variant="primary")
    status = gr.Textbox(label="Status", lines=3)
    
    gr.Markdown("### Game Chat")
    
    chat = gr.ChatInterface(
        fn=run_action,
        examples=[
            "start game",
            "Inventory",
            "look around",
            "talk to someone nearby",
            "explore the area",
            "spend 10 credits to bribe a local"
        ],
        title=None
    )
    
    # Connect load button
    load_btn.click(
        fn=setup_game,
        inputs=[planet_input, character_input],
        outputs=[status]
    )

In [ ]:
interface.launch(server_port=7860)